# Structuring Your Unstructured Data with AI SQL Functions

Most real-world data isn't clean rows and columns — it's free-form text, PDFs, and documents. In this notebook we'll use Databricks **AI SQL Functions** to turn that unstructured data into structured, queryable tables using a single SQL expression.

**Why batch inference matters:**
- **Analysts** get structured columns they can filter, aggregate, and visualize
- **Classical ML** gets clean features without manual labelling pipelines
- **Agentic systems** get structured memory and tool-ready outputs

**What we'll cover:**
1. AI SQL Functions overview
2. Structuring free-form text (`ai_classify`, `ai_extract`, `ai_query`)
3. Wrapping a custom prompt in a reusable Unity Catalog function
4. Parsing documents (`ai_parse` on a PDF)
5. Evaluating AI function outputs
6. Challenge activity

## 1. Notebook Setup

In [0]:
# DO NOT MODIFY - FOR DEMO ONLY
current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
unique_table_suffix = current_user.split('@')[0].replace('.', '_')

In [0]:
# create notebook widgets so we can pass dynamic parameters in jobs
dbutils.widgets.text("catalog", "classic_stable_been2c_catalog", "Catalog")
dbutils.widgets.text("schema", "weather", "Schema")
dbutils.widgets.text("username", unique_table_suffix, "Username")

In [0]:
# get widgets values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
username = dbutils.widgets.get("username")

print("catalog: ", catalog)
print("schema: ", schema)
print("username: ", username)

## 2. AI SQL Functions Overview

Databricks AI SQL Functions are built-in SQL functions that call foundation models directly inside a query. There's no model deployment, no API key management, and no separate inference pipeline — the model call happens inline, one row at a time, as part of your SQL or PySpark expression.

Under the hood they use the **Databricks Foundation Model API**, which is served through Unity Catalog and governed like any other data asset.

**Key properties:**
- Works in SQL, PySpark (`expr()`), and Databricks notebooks
- Scales to millions of rows via Spark parallelism
- Results can be written straight to a Delta table — making them available to dashboards, Genie, and downstream pipelines

**2a. `ai_classify` — Assign a label from a fixed list**

`ai_classify` picks the single best-matching label from the list you provide. It's deterministic and fast — ideal for categorisation tasks where you control the label set.

In [0]:
%sql
SELECT
    phrase_long,
    ai_classify(phrase_long, ARRAY('Clear', 'Cloudy', 'Precipitation', 'Severe')) AS weather_category
FROM 
    IDENTIFIER(:catalog || '.' || :schema || '.us_postal_daily_metric') -- create dynamic table name based on widget inputs
LIMIT 5;

**2b. `ai_extract` — Pull named attributes from text**

`ai_extract` reads free-form text and returns a struct of named fields. You define what you want to extract — the model figures out how to find it in the text.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_us_postal_daily_metric AS
SELECT
    phrase_long,
    ai_extract(
        phrase_long,
        ARRAY('precipitation_type', 'sky_condition', 'intensity')
    ) AS extracted
FROM 
    IDENTIFIER(:catalog || '.' || :schema || '.us_postal_daily_metric');

SELECT * FROM temp_us_postal_daily_metric LIMIT 5;

In [0]:
%sql
-- we can use dot notation to flatten our extracted column
SELECT 
  phrase_long,
  extracted.precipitation_type,
  extracted.sky_condition,
  extracted.intensity
FROM temp_us_postal_daily_metric
LIMIT 5;

**2c. `ai_query` — Custom prompt, maximum flexibility**

`ai_query` gives you a direct line to the model. You pass the model name and a full prompt string — useful when classify and extract don't cover your use case.

In [0]:
%sql
SELECT
    phrase_long,
    ai_query(
        'databricks-claude-sonnet-4', -- choose any foundation model
        "Is this weather condition suitable for flying a small aircraft? Condition: " || phrase_long -- this is our prompt
    ) AS flying_suitability
FROM 
    IDENTIFIER(:catalog || '.' || :schema || '.us_postal_daily_metric')
LIMIT 5;

## 3. Creating a resusable Unity Catalog registered User Defined Function (UDF)

Repeating a long prompt string in every notebook is fragile — if the prompt changes you have to find and update every reference. A better pattern is to wrap the `ai_query` call in a **Python function registered to Unity Catalog**.

Benefits:
- **Reusable** — call it from any notebook, SQL editor, or job with `catalog.schema.function_name()`
- **Governed** — versioned and permissioned like any other UC asset
- **Consistent** — the prompt logic lives in one place

The function below analyses a weather phrase and returns a **structured JSON string** with three fields: `suitable` (bool), `risk_level` (High / Medium / Low), and `reason` (one sentence). Returning JSON rather than plain text makes it easy to parse the output into columns downstream.

In [0]:
%sql
-- create a dynamic function name for the demo, you would typically hardcode this
CREATE OR REPLACE FUNCTION IDENTIFIER(:catalog || '.' || :schema || '.fx_small_aircraft_flight_suitability_' || :username)(phrase STRING)
  RETURNS STRING
  COMMENT "Determines flight suitability based on weather"
  RETURN 
    SELECT ai_query(
      "databricks-claude-sonnet-4",
      "Is this weather condition suitable for flying a small aircraft? Condition: " || phrase,
      responseFormat => '{
        "type": "json_schema",
        "json_schema": {
          "name": "small_aircraft_flight_suitability",
          "schema": {
            "type": "object",
            "properties": {
              "suitability": {"type": "string"},
              "reason": {"type": "string"}
            }
        },
        "strict": true
      }
      }
    }'
  );

**Call the registered UC function from SQL**

Once registered, anyone with `EXECUTE` permission on the function can call it by its three-part name.

In [0]:
%sql
SELECT
  phrase_long,
  IDENTIFIER(:catalog || '.' || :schema || '.fx_small_aircraft_flight_suitability_' || :username)(phrase_long) AS assessment
FROM 
  IDENTIFIER(:catalog || '.' || :schema || '.us_postal_daily_metric')
LIMIT 10;


## 4. Parsing Documents with `ai_parse_document`

So far we've worked with short text strings. But a lot of valuable information lives in documents — PDFs, reports, manuals. `ai_parse_document` reads a binary file from a Databricks Volume and extracts its content as structured text.

We'll use the **RCAF Weather Manual** — a document describing aviation weather standards — as our example.

In [0]:
%sql
SELECT
  ai_parse_document(content) AS parsed
FROM READ_FILES('/Volumes/' || :catalog || '/' || :schema || '/files/RCAF-Weather-Manual.pdf', format => 'binaryFile')

**We can create a single column of text content**

In [0]:
%sql
WITH parsed_documents AS (
    SELECT
      path,
      ai_parse_document(
        content
      ) AS parsed
    FROM READ_FILES('/Volumes/' || :catalog || '/' || :schema || '/files/RCAF-Weather-Manual.pdf', format => 'binaryFile')
  ),
  
  parsed_text AS (
    SELECT
      path,
      concat_ws(
        '\n\n',
        transform(
          try_cast(parsed:document:elements AS ARRAY<VARIANT>),
          element -> try_cast(element:content AS STRING)
        )
      ) AS text
    FROM parsed_documents
    WHERE try_cast(parsed:error_status AS STRING) IS NULL
  )

SELECT * FROM parsed_text;

## 5. Evaluating AI Function Outputs

AI SQL functions are powerful, but their outputs are probabilistic — the same input can produce slightly different results, and the model can make mistakes. Before trusting batch inference results in a production pipeline, it's worth thinking about evaluation.

**Three-step approach:**

**1. Create a labelled benchmark**
Hand-label 50–100 rows with the correct answer. This becomes your ground truth. Keep it small enough to do manually, but large enough to be statistically meaningful. Store it as a Delta table.

**2. Run your AI function against the benchmark**
Apply `ai_classify`, `ai_extract`, or your UDF to the benchmark rows and compare the outputs to your labels. Calculate accuracy, precision, recall, or whatever metric fits your use case.

**3. Use `ai_query` as an LLM judge**
For tasks where there's no single right answer (e.g. free-text generation), use a second `ai_query` call to evaluate the quality of the first. The judge prompt asks the model to score or compare outputs. This is called **LLM-as-a-judge**.

```sql
-- Example: use ai_query to judge whether a classification was correct
SELECT
    phrase_long,
    predicted_category,
    ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT(
            'A weather classifier labelled the phrase "', phrase_long, '" as "', predicted_category, '". ',
            'Is this label correct? Answer only: Correct or Incorrect.'
        )
    ) AS judge_verdict
FROM benchmark_results
```

**Practical tips:**
- Re-evaluate after prompt changes — small wording changes can shift accuracy meaningfully
- Track agreement rate (% where the judge agrees with your label) over time as a quality signal
- For production pipelines, fail the job or alert if accuracy drops below a threshold